# Model Inference Analysis — SHAP and PDP


## 1. Imports and Configuration


In [ ]:
# Core libraries
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.inspection import PartialDependenceDisplay

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

MODEL_ID = 'xgboost'
RUN_NAME = "nusc_mini_debug_tpp-11_Mar_2026_15_29_02"
TARGET_COL = None  # Optional override, e.g. 'ml_ade_log'

RESULTS_ROOT = Path('../../results/interpretable_model') / MODEL_ID / RUN_NAME
TABLES_DIR = RESULTS_ROOT / 'tables'
PLOTS_DIR = RESULTS_ROOT / 'plots'

print(f'Results root: {RESULTS_ROOT.resolve()}')
print(f'Model ID: {MODEL_ID}')
print(f'Run: {RUN_NAME}')
print(f'TARGET_COL override: {TARGET_COL}')


## 2. Load Run Manifest and Final-Model Artifacts


In [ ]:
def resolve_manifest_path(model_id, run_name, target_col=None):
    manifest_dir = Path('../../results/interpretable_model') / model_id / run_name / 'tables'
    if not manifest_dir.exists():
        raise FileNotFoundError(f'No tables directory found for model_id={model_id}, run_name={run_name}: {manifest_dir}')

    if target_col is not None:
        manifest_path = manifest_dir / f'run_manifest_{target_col}.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'Run manifest not found for target_col={target_col}: {manifest_path}')
        return manifest_path

    manifest_candidates = sorted(manifest_dir.glob('run_manifest_*.json'))
    if not manifest_candidates:
        raise FileNotFoundError(f'No run_manifest_*.json files found in {manifest_dir}')
    if len(manifest_candidates) > 1:
        raise ValueError(
            f'Multiple run manifests found in {manifest_dir}. Set TARGET_COL explicitly. '
            f'Candidates: {[p.name for p in manifest_candidates]}'
        )
    return manifest_candidates[0]


manifest_path = resolve_manifest_path(MODEL_ID, RUN_NAME, TARGET_COL)
manifest = json.loads(manifest_path.read_text())

if manifest['model_id'] != MODEL_ID:
    raise ValueError(f"Manifest model_id={manifest['model_id']} does not match MODEL_ID={MODEL_ID}")
if manifest['model_id'] != 'xgboost':
    raise NotImplementedError(f"Model inference analysis is not implemented yet for model_id={manifest['model_id']}")

target_col = manifest['target_col']
feature_cols = manifest['feature_cols']
TABLES_DIR = Path(manifest['tables_dir'])
PLOTS_DIR = Path(manifest['plots_dir'])

nested_resampling = manifest['nested_resampling']
final_model = manifest['final_model']
model_data_path = Path(nested_resampling['model_data_with_oof_path'])
model_path = Path(final_model['model_path'])
full_data_tuning_summary_path = Path(final_model['full_data_tuning_summary_path'])
full_data_tuning_summary = json.loads(full_data_tuning_summary_path.read_text())

model_df_oof = pd.read_csv(model_data_path)
X = model_df_oof[feature_cols]

model = xgb.XGBRegressor()
model.load_model(model_path)

print(f'Loaded manifest: {manifest_path}')
print(f'Loaded model data: {model_data_path}')
print(f'Loaded final model: {model_path}')
print(f'Target: {target_col} | Features: {len(feature_cols)}')
print('Final-model tuning summary:')
print(json.dumps(full_data_tuning_summary, indent=2))


## 3. SHAP Interpretability (Global + Local)


In [ ]:
explainer = shap.Explainer(model, X)
shap_exp = explainer(X)
shap_values = shap_exp.values

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    'feature': X.columns,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

display(shap_importance_df)

shap_imp_path = TABLES_DIR / f'shap_importance_{target_col}.csv'
shap_importance_df.to_csv(shap_imp_path, index=False)
print(f'SHAP importance table saved to: {shap_imp_path}')

top_features = shap_importance_df['feature'].head(6).tolist()
print('Top 6 features by mean |SHAP|:')
print(top_features)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, show=False, max_display=20)
beeswarm_path = PLOTS_DIR / f'shap_beeswarm_{target_col}.png'
plt.tight_layout()
plt.savefig(beeswarm_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, plot_type='bar', show=False, max_display=20)
bar_path = PLOTS_DIR / f'shap_bar_{target_col}.png'
plt.tight_layout()
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f'SHAP beeswarm saved to: {beeswarm_path}')
print(f'SHAP bar saved to:      {bar_path}')


## 4. Partial Dependence Plots (Top 6 SHAP Features)


In [ ]:
for feat in top_features:
    fig, ax = plt.subplots(figsize=(8, 5))
    PartialDependenceDisplay.from_estimator(
        model,
        X,
        features=[feat],
        kind='average',
        grid_resolution=40,
        ax=ax,
    )
    ax.set_title(f'PDP — {feat}')
    pdp_path = PLOTS_DIR / f'pdp_{feat}_{target_col}.png'
    plt.tight_layout()
    plt.savefig(pdp_path, dpi=150, bbox_inches='tight')
    plt.show()

print('PDP plots saved for top 6 SHAP-ranked features.')


## 5. Saved Artifacts


In [ ]:
print('Saved artifacts:')
print(f'- Run manifest:            {manifest_path}')
print(f'- Final model:             {model_path}')
print(f'- Tuning summary:          {full_data_tuning_summary_path}')
print(f'- SHAP importance table:   {shap_imp_path}')
print(f'- Plot directory:          {PLOTS_DIR}')
